# InsightForge AI — Agent 4: Visualization Agent
Selects and builds interactive Plotly charts based on column datatypes and distributions.


In [ ]:
%pip install pandas plotly
dbutils.library.restartPython()


In [ ]:
class InsightForgeState(TypedDict):
    """
    Shared state passed through all agents in the pipeline.
    Each agent reads what it needs and writes its output.
    No agent modifies another agent's output fields.

    Fields
    ------
    dataset_path     : path to the input CSV file
    gemini_key       : Gemini API key passed at runtime
    raw_df           : original DataFrame as uploaded
    cleaned_df       : DataFrame after cleaning agent runs
    schema_info      : column metadata detected by schema agent
    cleaning_report  : summary of all cleaning actions taken
    eda_results      : statistical analysis from EDA agent
    charts           : list of Plotly figure dicts from viz agent
    insights         : AI generated business insights text
    pdf_path         : path to the generated PDF report
    pipeline_log     : timestamped log of each agent execution
    errors           : list of error messages from any agent
    """
    dataset_path    : str
    gemini_key      : str
    raw_df          : Any
    cleaned_df      : Any
    schema_info     : dict
    cleaning_report : dict
    eda_results     : dict
    charts          : list
    insights        : str
    pdf_path        : str
    pipeline_log    : list
    errors          : list

print("✅ InsightForgeState defined")
print()
print("  State fields:")
fields = [
    ("dataset_path",     "input — path to CSV"),
    ("gemini_key",       "input — API key"),
    ("raw_df",           "Schema Agent reads this"),
    ("cleaned_df",       "Cleaning Agent writes this"),
    ("schema_info",      "Schema Agent writes this"),
    ("cleaning_report",  "Cleaning Agent writes this"),
    ("eda_results",      "EDA Agent writes this"),
    ("charts",           "Visualization Agent writes this"),
    ("insights",         "Insight Agent writes this"),
    ("pdf_path",         "Report Agent writes this"),
    ("pipeline_log",     "every agent appends to this"),
    ("errors",           "every agent appends on failure"),
]
for field, desc in fields:
    print(f"    {field:20} — {desc}")


In [ ]:
def log_event(state: InsightForgeState, agent: str, message: str) -> list:
    """
    Appends a timestamped log entry to the pipeline log.
    Called by every agent on start and completion.

    Parameters
    ----------
    state   : current pipeline state
    agent   : name of the calling agent
    message : what happened

    Returns
    -------
    list : updated pipeline log
    """
    timestamp = datetime.now().strftime("%H:%M:%S")
    entry     = f"[{timestamp}] {agent}: {message}"
    print(f"   {entry}")
    return state["pipeline_log"] + [entry]


def get_gemini_model() -> genai.GenerativeModel:
    """
    Returns a configured Gemini model instance.
    Re-reads the API key from widget each time to handle
    session restarts without needing to re-run setup cells.
    """
    key = dbutils.widgets.get("gemini_key")
    genai.configure(api_key=key)
    return genai.GenerativeModel(GEMINI_MODEL)


def safe_call_gemini(prompt: str, agent_name: str) -> str:
    """
    Wraps a Gemini API call with error handling.
    Cleans the response text to remove characters that
    fpdf2 cannot render with standard Helvetica font.

    Parameters
    ----------
    prompt     : the full prompt string to send
    agent_name : name of the calling agent for logging

    Returns
    -------
    str : cleaned response text or error message
    """
    try:
        m        = get_gemini_model()
        response = m.generate_content(prompt)
        text     = response.text

        # Remove characters unsupported by Helvetica in fpdf2
        replacements = {
            "\u2014": "-",    # em dash
            "\u2013": "-",    # en dash
            "\u2012": "-",    # figure dash
            "\u2011": "-",    # non-breaking hyphen
            "\u2010": "-",    # hyphen
            "\u2022": "-",    # bullet
            "\u2023": "-",    # triangle bullet
            "\u2043": "-",    # hyphen bullet
            "\u2018": "'",    # left single quote
            "\u2019": "'",    # right single quote
            "\u201a": "'",    # single low quote
            "\u201c": '"',    # left double quote
            "\u201d": '"',    # right double quote
            "\u201e": '"',    # double low quote
            "\u2026": "...",  # ellipsis
            "\u00a0": " ",    # non-breaking space
            "\u00b7": "-",    # middle dot
            "\u2015": "-",    # horizontal bar
        }
        for char, replacement in replacements.items():
            text = text.replace(char, replacement)

        # Final safety pass — replace remaining non-latin-1 chars
        text = text.encode("latin-1", errors="replace").decode("latin-1")
        return text

    except Exception as e:
        logger.warning(
            f"Gemini call failed in {agent_name}: {str(e)[:100]}"
        )
        print(f"   ⚠️  Gemini call failed in {agent_name}: {e}")
        return f"[Gemini error in {agent_name}: {str(e)}]"


print("✅ Utility functions defined")
print("   log_event()        — timestamped pipeline logging")
print("   get_gemini_model() — safe model initialisation")
print("   safe_call_gemini() — error handled API call with font cleaning")


In [ ]:
def viz_agent(state: InsightForgeState) -> dict:
    """
    Agent 4 — Visualization Agent
    ------------------------------
    Reads  : state["cleaned_df"], state["schema_info"],
             state["eda_results"]
    Writes : state["charts"]
    """
    agent_name = "Visualization Agent"
    print(f"\n{'─' * 55}")
    print(f"🟣 {agent_name} starting...")

    df     = state["cleaned_df"]
    schema = state["schema_info"]
    eda    = state["eda_results"]
    errors = state["errors"]
    log    = log_event(state, agent_name, "started")

    target       = schema.get("target_variable", "")
    numeric_cols = [
        c for c in eda.get("numeric_cols", []) if c in df.columns
    ]
    cat_cols = [
        c for c in eda.get("categorical_cols", []) if c in df.columns
    ]

    logger.info(
        f"{agent_name} started — "
        f"{len(numeric_cols)} numeric, {len(cat_cols)} categorical"
    )

    charts = []

    try:
        for col in numeric_cols[:4]:
            fig = px.histogram(
                df, x=col,
                title                   = f"Distribution - {col}",
                template                = "plotly_white",
                color_discrete_sequence = ["#4C72B0"]
            )
            fig.update_layout(xaxis_title=col, yaxis_title="Count")
            charts.append({"type":"histogram","title":f"Distribution - {col}","col":col,"fig":fig})
            fig.show()
            print(f"   Chart: histogram — {col}")

        for col in cat_cols[:3]:
            vc = df[col].value_counts().head(10).reset_index()
            vc.columns = [col, "Count"]
            fig = px.bar(
                vc, x=col, y="Count",
                title    = f"Value Counts - {col}",
                template = "plotly_white",
                color    = col, text="Count"
            )
            fig.update_traces(textposition="outside")
            fig.update_layout(showlegend=False)
            charts.append({"type":"bar","title":f"Value Counts - {col}","col":col,"fig":fig})
            fig.show()
            print(f"   Chart: bar — {col}")

        if len(numeric_cols) >= 2:
            corr = df[numeric_cols].corr().round(2)
            fig  = px.imshow(
                corr, text_auto=True,
                title                  = "Correlation Heatmap",
                color_continuous_scale = "RdBu_r",
                template               = "plotly_white",
                zmin=-1, zmax=1
            )
            fig.update_layout(width=650, height=500)
            charts.append({"type":"heatmap","title":"Correlation Heatmap","col":"all","fig":fig})
            fig.show()
            print(f"   Chart: correlation heatmap")

        if target and target in df.columns:
            for col in cat_cols[:2]:
                group = (
                    df.groupby(col)[target]
                    .mean()
                    .reset_index()
                    .rename(columns={target: f"{target} Rate"})
                )
                group[f"{target} Rate"] = group[f"{target} Rate"].round(4)
                fig = px.bar(
                    group, x=col, y=f"{target} Rate",
                    title    = f"{target} Rate by {col}",
                    template = "plotly_white",
                    color=col, text=f"{target} Rate"
                )
                fig.update_traces(textposition="outside")
                fig.update_layout(yaxis_range=[0,1.15], showlegend=False)
                charts.append({"type":"bar","title":f"{target} Rate by {col}","col":col,"fig":fig})
                fig.show()
                print(f"   Chart: {target} rate by {col}")

        if "Age" in df.columns and "Fare" in df.columns:
            color_col = target if target in df.columns else None
            fig = px.scatter(
                df, x="Age", y="Fare", color=color_col,
                title    = f"Age vs Fare{' - coloured by '+target if color_col else ''}",
                template = "plotly_white", opacity=0.65
            )
            fig.update_traces(marker=dict(size=5))
            charts.append({"type":"scatter","title":"Age vs Fare","col":"Age_Fare","fig":fig})
            fig.show()
            print(f"   Chart: scatter — Age vs Fare")

        log = log_event(
            state, agent_name,
            f"done — {len(charts)} charts generated"
        )

        logger.info(f"{agent_name} complete — {len(charts)} charts generated")

        print(f"   Total : {len(charts)} charts")
        print(f"✅ {agent_name} complete")

        return {
            "charts"       : charts,
            "pipeline_log" : log,
            "errors"       : errors
        }

    except Exception as e:
        msg = f"{agent_name} failed: {str(e)}"
        logger.error(f"{agent_name} FAILED — {str(e)}")
        print(f"   ❌ {msg}")
        return {
            "charts"       : [],
            "pipeline_log" : log_event(state, agent_name, f"FAILED — {e}"),
            "errors"       : errors + [msg]
        }

print("✅ viz_agent() defined")
